# **PoC RF-IA-01: Predicción de Estructura, Trazabilidad y Contexto Normativo para el Café de Chanchamayo**

 HU Asociada: HU-05 (Registro de información de producción de lote) y HU-08 (Registro de eventos de trazabilidad mediante Blockchain).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Cargar el dataset desde el almacenamiento de Colab
nombre_archivo = 'dataset_trazabilidad_cafe_60_lotes.xlsx'
df = pd.read_excel(nombre_archivo)

# 2. Asignación de Variables de Entrada (X) y Salidas Esperadas (y)
X = df[['altitud_msnm', 'variedad_cafe', 'metodo_beneficio', 'humedad_porcentaje', 'puntuacion_catacion']]
y_calidad = df['calidad_tipo']
y_etapa = df['etapa_destino']
y_hash = df['hash_blockchain_valid']

# 3. Preprocesamiento de Variables Numéricas y Categóricas
numeric_features = ['altitud_msnm', 'humedad_porcentaje', 'puntuacion_catacion']
categorical_features = ['variedad_cafe', 'metodo_beneficio']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# 4. División de Datos: 80% Entrenamiento (48 casos) / 20% Prueba (12 casos)
X_train, X_test, y_train_cal, y_test_cal = train_test_split(X, y_calidad, test_size=0.2, random_state=42)
_, _, y_train_etapa, y_test_etapa = train_test_split(X, y_etapa, test_size=0.2, random_state=42)
_, _, y_train_hash, y_test_hash = train_test_split(X, y_hash, test_size=0.2, random_state=42)

# 5. Entrenamiento de Clasificadores Predictivos
model_calidad = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
model_etapa = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])
model_hash = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', RandomForestClassifier(random_state=42))])

model_calidad.fit(X_train, y_train_cal)
model_etapa.fit(X_train, y_train_etapa)
model_hash.fit(X_train, y_train_hash)

# 6. Ejecución de Predicciones sobre los 12 casos de prueba
pred_calidad = model_calidad.predict(X_test)
pred_etapa = model_etapa.predict(X_test)
pred_hash = model_hash.predict(X_test)

# 7. Cálculo de Métricas Evaluadas sobre datos de prueba reales
acc_calidad = accuracy_score(y_test_cal, pred_calidad)
f1_calidad = f1_score(y_test_cal, pred_calidad, average='macro')
acc_etapa = accuracy_score(y_test_etapa, pred_etapa)
f1_hash = f1_score(y_test_hash, pred_hash, pos_label='Valido', average='micro')

# 8. Metas y Umbrales
meta_acc_calidad = 0.80
meta_f1_calidad = 0.75
meta_acc_etapa = 0.80
meta_f1_hash = 0.75

cumple_acc_calidad = "SI" if acc_calidad >= meta_acc_calidad else "NO"
cumple_f1_calidad = "SI" if f1_calidad >= meta_f1_calidad else "NO"
cumple_acc_etapa = "SI" if acc_etapa >= meta_acc_etapa else "NO"
cumple_f1_hash = "SI" if f1_hash >= meta_f1_hash else "NO"

todas_cumplen = all([
    acc_calidad >= meta_acc_calidad,
    f1_calidad >= meta_f1_calidad,
    acc_etapa >= meta_acc_etapa,
    f1_hash >= meta_f1_hash
])

decision_global = "GO - CUMPLE RF-IA-01" if todas_cumplen else "NO GO - NO CUMPLE RF-IA-01"

# 9. IMPRIMIR PREDICCIONES DETALLADAS CASO POR CASO
df_prueba = pd.DataFrame({
    'lote_id': df.loc[X_test.index, 'lote_id'],
    'calidad_real': y_test_cal.values,
    'calidad_predicha': pred_calidad,
    'etapa_real': y_test_etapa.values,
    'etapa_predicha': pred_etapa,
    'hash_real': y_test_hash.values,
    'hash_predicho': pred_hash
})

print("="*110)
print("                       PREDICCIONES DETALLADAS SOBRE EL CONJUNTO DE PRUEBA")
print("="*110)
print(df_prueba.to_string(index=False))
print("\n")

# 10. IMPRIMIR CUADRO RESUMEN DE MÉTRICAS Y DECISIÓN GLOBAL
df_resultados = pd.DataFrame({
    'Metrica': [
        'Accuracy calidad_tipo',
        'F1 macro calidad_tipo',
        'Accuracy etapa_destino',
        'F1 micro hash_blockchain_valid'
    ],
    'Resultado': [round(acc_calidad, 4), round(f1_calidad, 4), round(acc_etapa, 4), round(f1_hash, 4)],
    'Meta': [meta_acc_calidad, meta_f1_calidad, meta_acc_etapa, meta_f1_hash],
    'Cumple': [cumple_acc_calidad, cumple_f1_calidad, cumple_acc_etapa, cumple_f1_hash]
})

print("="*65)
print("            RESULTADOS DE LA PoC RF-IA-01")
print("="*65)
print(df_resultados.to_string(index=False))
print("="*65)
print(f"DECISIÓN GLOBAL: {decision_global}")
print("="*65)

                       PREDICCIONES DETALLADAS SOBRE EL CONJUNTO DE PRUEBA
 lote_id calidad_real calidad_predicha    etapa_real etapa_predicha hash_real hash_predicho
LOTE-101     Especial         Especial      Traslado    Exportacion    Valido        Valido
LOTE-106    Comercial        Comercial Procesamiento  Procesamiento  Invalido      Invalido
LOTE-137      Premium          Premium      Traslado       Traslado    Valido        Valido
LOTE-146      Premium          Premium      Traslado       Traslado    Valido        Valido
LOTE-114      Premium          Premium      Traslado       Traslado    Valido        Valido
LOTE-155      Premium          Premium      Traslado       Traslado    Valido        Valido
LOTE-134      Premium          Premium      Traslado       Traslado    Valido        Valido
LOTE-149      Premium          Premium      Traslado       Traslado    Valido        Valido
LOTE-113     Especial         Especial   Exportacion    Exportacion    Valido        Valido
LOTE-

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1618: UserWarning: Note that pos_label (set to 'Valido') is ignored when average != 'binary' (got 'micro'). You may use labels=[pos_label] to specify a single positive class.
  warnings.warn(
